# CoughTB — Inference with Pre-trained Model

ใช้ AI วิเคราะห์เสียงไอว่าเป็น **TB** หรือไม่

**Model**: MobileNetV4 + Res2TSM (train บน CODA TB DREAM Challenge)

**Source**: https://github.com/yop-dev/tb-cough-detection

---
### วิธีใช้
1. รัน cell ทั้งหมด
2. อัปโหลดไฟล์ .wav (หรือ .mp3/.flac)
3. AI จะแสดงผลว่า TB หรือไม่ + Confidence


## 1. ติดตั้ง Dependencies

In [ ]:
!pip install -q torch torchaudio timm librosa soundfile numpy matplotlib pillow scipy huggingface_hub

## 2. Clone repo (for model weights only) + กำหนด Model Architecture

In [ ]:
import os
if not os.path.exists("tb-cough-detection"):
    !git clone https://github.com/yop-dev/tb-cough-detection.git

import torch
import torch.nn as nn
import timm
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# --- Model Architecture (from yop-dev/tb-cough-detection) ---

class TemporalShift(nn.Module):
    def __init__(self, channels, shift_div=8):
        super().__init__()
        self.fold = channels // shift_div
    def forward(self, x):
        B, C, T, F = x.size()
        t = x.permute(0, 2, 1, 3).contiguous()
        out = torch.zeros_like(t)
        out[:, :-1, :self.fold, :] = t[:, 1:, :self.fold, :]
        out[:, 1:, self.fold:2*self.fold, :] = t[:, :-1, self.fold:2*self.fold, :]
        out[:, :, 2*self.fold:, :] = t[:, :, 2*self.fold:, :]
        return out.permute(0, 2, 1, 3)

class Res2TSMBlock(nn.Module):
    def __init__(self, channels, scale=4, shift_div=8):
        super().__init__()
        self.scale = scale
        self.width = channels // scale
        self.temporal_shift = TemporalShift(channels, shift_div)
        self.convs = nn.ModuleList([
            nn.Conv2d(self.width, self.width, kernel_size=(3, 1),
                      padding=(1, 0), groups=self.width, bias=False)
            for _ in range(scale-1)
        ])
        self.bn = nn.BatchNorm2d(channels)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.temporal_shift(x)
        splits = torch.split(x, self.width, dim=1)
        y = splits[0]
        outs = [y]
        for i in range(1, self.scale):
            sp = splits[i] + y
            sp = self.convs[i-1](sp)
            y = sp
            outs.append(sp)
        out = torch.cat(outs, dim=1)
        return self.act(self.bn(out))

class MobileNetV4_Res2TSM(nn.Module):
    def __init__(self, model_key, scale=4, shift_div=8, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_key, pretrained=False, features_only=True)
        C = self.backbone.feature_info.channels()[-1]
        self.res2tsm = Res2TSMBlock(C, scale=scale, shift_div=shift_div)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(C, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        feat = self.backbone(x)[-1]
        feat = self.res2tsm(feat)
        out = self.global_pool(feat).view(feat.size(0), -1)
        return self.fc(out).squeeze(1)

# --- Load Pre-trained Weights ---

model = MobileNetV4_Res2TSM('mobilenetv4_conv_blur_medium').to(DEVICE)

model_path = "tb-cough-detection/final_best_mobilenetv4_conv_blur_medium_res2tsm_tb_classifier.pth"
state = torch.load(model_path, map_location=DEVICE, weights_only=True)

if 'state_dict' in state:
    sd = state['state_dict']
elif 'model_state_dict' in state:
    sd = state['model_state_dict']
else:
    sd = state

sd = {k.replace('module.', ''): v for k, v in sd.items()}
model.load_state_dict(sd, strict=False)
model.eval()

print("Model loaded successfully!")
print(f"  Architecture: MobileNetV4 + Res2TSM")
print(f"  Input: Mel spectrogram 224x224x3")

## 3. ดาวน์โหลด CODA TB Dataset จาก HuggingFace

ดาวน์โหลดไฟล์เสียงจริงของคนไข้ TB+ และ TB-


In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

print("=" * 60)
print("Downloading CODA TB dataset from HuggingFace...")
print("=" * 60)

zip_path = hf_hub_download(
    repo_id="AHFIDAILabs/coda_tb_dataset",
    filename="dataset.zip",
    repo_type="dataset",
)
print(f"Downloaded: {zip_path}")

extract_dir = "coda_tb"
Path(extract_dir).mkdir(exist_ok=True)

print("Extracting... (may take a few minutes)")
with zipfile.ZipFile(zip_path, "r") as z:
    for member in z.infolist():
        if not member.filename or member.filename.endswith("/"):
            continue
        parts = member.filename.split("/")
        if "solicited_data" in parts:
            idx = parts.index("solicited_data")
            member.filename = f"solicited_data/{parts[-1]}"
        elif "longitudinal_data" in parts:
            idx = parts.index("longitudinal_data")
            member.filename = f"longitudinal_data/{parts[-1]}"
        else:
            member.filename = "/".join(parts[1:])
        try:
            z.extract(member, extract_dir)
        except:
            pass
print("Extraction complete!")

for subdir in ["solicited_data", "longitudinal_data", "meta_data"]:
    p = Path(extract_dir) / subdir
    if p.exists():
        files = list(p.iterdir())
        print(f"{subdir}: {len(files)} files")
        if files:
            print(f"  Example: {files[0].name}")

## 3.1 ฟังก์ชัน Inference



รับไฟล์เสียง → วิเคราะห์ → คืนค่า probability (0=ไม่เป็น, 1=เป็น TB)

In [ ]:
import librosa
import numpy as np
import torch
from scipy.ndimage import zoom

SR = 16000
CROP = 0.5  # seconds
N_MELS = 224
FMAX = 8000

def load_and_segment(path):
    y, _ = librosa.load(path, sr=SR)
    if len(y) == 0:
        return None
    y, _ = librosa.effects.trim(y, top_db=20)
    target_len = int(SR * CROP)
    if len(y) >= target_len:
        energy = np.convolve(y**2, np.ones(target_len), mode='valid')
        start = np.argmax(energy)
        seg = y[start:start+target_len]
    else:
        seg = np.zeros(target_len, dtype=y.dtype)
        seg[:len(y)] = y
    return seg

def make_mel_rgb(y_seg):
    mel = librosa.feature.melspectrogram(
        y=y_seg, sr=SR, n_mels=N_MELS, fmax=FMAX,
        hop_length=512, win_length=2048
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    target_shape = (224, 224)
    if mel_db.shape != target_shape:
        zf = (target_shape[0]/mel_db.shape[0], target_shape[1]/mel_db.shape[1])
        resized = zoom(mel_db, zf, order=1)
    else:
        resized = mel_db

    if np.ptp(resized) > 0:
        normed = (resized - resized.min()) / np.ptp(resized)
    else:
        normed = np.zeros_like(resized)

    rgb = np.stack([normed] * 3, axis=0)
    return mel_db, rgb

def predict_cough(file_path):
    """
    วิเคราะห์เสียงไอ
    คืนค่า: probability (0-1), ป้ายผล
    """
    y_seg = load_and_segment(file_path)
    if y_seg is None:
        return None, "Error: cannot load audio"

    _, mel_rgb = make_mel_rgb(y_seg)
    if mel_rgb is None:
        return None, "Error: cannot process audio"

    with torch.no_grad():
        input_tensor = torch.from_numpy(mel_rgb).float().unsqueeze(0).to(DEVICE)
        prob = model(input_tensor).cpu().numpy()[0]

    if prob > 0.5:
        label = f"⚠️  TB DETECTED  (confidence: {prob*100:.1f}%)"
    else:
        label = f"✅ No TB detected  (confidence: {(1-prob)*100:.1f}%)"

    return prob, label

print("ฟังก์ชันพร้อมทำงาน")

In [ ]:
import pandas as pd
import glob

csv_files = list(Path("coda_tb").rglob("*.csv"))
print(f"Found {len(csv_files)} CSV files")

meta = {}
for f in csv_files:
    df = pd.read_csv(f)
    meta[f.stem] = df
    print(f"\n{f.stem}: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    if 'tb_status' in df.columns:
        print(f"  TB distribution:\n{df['tb_status'].value_counts().to_string()}")

### 3.2 Evaluation — วัดความแม่นยำของ Model

ทำ patient-level split → ทดสอบ → confusion matrix + metrics

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score, roc_curve
)
import matplotlib.pyplot as plt

# Support both local path and Colab download
LOCAL_PATH = Path(r"C:\Users\kongw\OneDrive\Documents\PROJECT\CoughTB\dataset\dataset")
if LOCAL_PATH.exists():
    base_dir = LOCAL_PATH
    print(f"Using local dataset: {base_dir}")
else:
    base_dir = Path("coda_tb")
    print(f"Using Colab dataset: {base_dir}")

solicited_dir = base_dir / "raw_data" / "solicited_data"
if not solicited_dir.exists():
    solicited_dir = base_dir / "solicited_data"
wav_files = sorted(solicited_dir.glob("*.wav"))
print(f"WAV files: {len(wav_files)}")

# Load Clinical metadata (has tb_status) and Solicited metadata (has WAV filenames)
clinical_path = base_dir / "meta_data" / "Clinical" / "CODA_TB_Clinical_Meta_Info.csv"
solicited_meta_path = base_dir / "meta_data" / "Cough Metadata" / "CODA_TB_Solicited_Meta_Info.csv"

# Fallback patterns if structure differs
if not clinical_path.exists():
    clinical_path = next(Path(base_dir).rglob("*Clinical_Meta_Info*"), None)
if not solicited_meta_path.exists():
    solicited_meta_path = next(Path(base_dir).rglob("*Solicited_Meta_Info*"), None)

print(f"\nClinical metadata: {clinical_path.name if clinical_path else 'NOT FOUND'}")
print(f"Solicited metadata: {solicited_meta_path.name if solicited_meta_path else 'NOT FOUND'}")

clinical = pd.read_csv(clinical_path)
solicited_meta = pd.read_csv(solicited_meta_path)

print(f"\nClinical: {clinical.shape} — columns: {list(clinical.columns)}")
print(f"Solicited metadata: {solicited_meta.shape} — columns: {list(solicited_meta.columns)}")
print(f"TB distribution:\n{clinical['tb_status'].value_counts().to_string()}")

# Merge on participant ID
df = solicited_meta.merge(clinical[['participant', 'tb_status']], on='participant')
print(f"\nMerged: {df.shape}")
print(f"TB+: {(df['tb_status']==1).sum()}, TB-: {(df['tb_status']==0).sum()}")

# Check WAV files exist
df['_wav_exists'] = df['filename'].apply(lambda x: (solicited_dir / x).exists())
available = df[df['_wav_exists']].copy()
print(f"WAVs found: {available['_wav_exists'].sum()}/{len(df)}")


In [ ]:
# Patient-level split + Inference on test set

TEST_SIZE = 0.2
RANDOM_STATE = 42

unique_patients = available[['participant', 'tb_status']].drop_duplicates(subset=['participant'])
train_patients, test_patients = train_test_split(
    unique_patients, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=unique_patients['tb_status']
)
train_ids = set(train_patients['participant'])
test_ids = set(test_patients['participant'])

train_set = available[available['participant'].isin(train_ids)]
test_set = available[available['participant'].isin(test_ids)]

print(f"Train patients: {len(train_ids)}, WAV files: {len(train_set)}")
print(f"Test patients: {len(test_ids)}, WAV files: {len(test_set)}")
print(f"Test TB+: {(test_set['tb_status']==1).sum()}, TB-: {(test_set['tb_status']==0).sum()}")


In [ ]:
# Run inference on test set

y_true, y_prob = [], []

for _, row in test_set.iterrows():
    wav_path = solicited_dir / row['filename']
    if not wav_path.exists():
        continue
    prob, _ = predict_cough(str(wav_path))
    if prob is not None:
        y_prob.append(prob)
        y_true.append(row['tb_status'])

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob > 0.5).astype(int)

print(f"\nEvaluated on {len(y_true)} WAV files")
print(f"Actual TB+: {y_true.sum()}, TB-: {(1-y_true).sum()}")


In [ ]:
# Confusion Matrix & Metrics

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0

print("=" * 55)
print("           EVALUATION RESULTS")
print("=" * 55)
print(f"\n  Confusion Matrix:")
print(f"                     Predicted")
print(f"                    Neg    Pos")
print(f"  Actual  Negative  {tn:>4}  {fp:>4}")
print(f"          Positive  {fn:>4}  {tp:>4}")
print(f"\n  Sensitivity (Recall): {sensitivity:.3f}  ({sensitivity*100:.1f}%)")
print(f"  Specificity:          {specificity:.3f}  ({specificity*100:.1f}%)")
print(f"  Accuracy:             {accuracy:.3f}  ({accuracy*100:.1f}%)")
print(f"  Precision:            {precision:.3f}  ({precision*100:.1f}%)")
print(f"  F1 Score:             {f1:.3f}")
print(f"  ROC-AUC:              {auc:.3f}")
print("=" * 55)

In [ ]:
# ROC Curve

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
youden_idx = np.argmax(tpr - fpr)
best_thresh = thresholds[youden_idx]
print(f"Youden's J optimal threshold: {best_thresh:.3f}")
print(f"  At threshold: Sensitivity={tpr[youden_idx]:.3f}, Specificity={1-fpr[youden_idx]:.3f}")

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
plt.plot(fpr[youden_idx], tpr[youden_idx], 'ro', markersize=8,
         label=f'Youden: thresh={best_thresh:.2f}')
plt.xlabel('1 - Specificity (FPR)')
plt.ylabel('Sensitivity (TPR)')
plt.title('ROC Curve — CoughTB on CODA TB Test Set')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Per-patient aggregation
test_set = test_set.copy()
test_set['_prob'] = np.nan
idx = 0
for i, row in test_set.iterrows():
    wav_path = solicited_dir / row['filename']
    if wav_path.exists() and idx < len(y_prob):
        test_set.at[i, '_prob'] = y_prob[idx]
        idx += 1

patient_results = test_set.groupby('participant').agg(
    mean_prob=('_prob', 'mean'),
    tb_status=('tb_status', 'first'),
    n_coughs=('filename', 'count')
).dropna()

patient_results['pred'] = (patient_results['mean_prob'] > 0.5).astype(int)

p_tn, p_fp, p_fn, p_tp = confusion_matrix(
    patient_results['tb_status'], patient_results['pred']
).ravel()

print(f"\n{'='*55}")
print(f"  PER-PATIENT RESULTS ({len(patient_results)} patients)")
print(f"{'='*55}")
print(f"  Sensitivity: {p_tp/(p_tp+p_fn)*100:.1f}%")
print(f"  Specificity: {p_tn/(p_tn+p_fp)*100:.1f}%")
print(f"  Accuracy:    {(p_tp+p_tn)/(p_tp+p_tn+p_fp+p_fn)*100:.1f}%")


## 4. ทดสอบกับไฟล์ตัวอย่าง

รัน cell นี้เพื่ออัปโหลดไฟล์เสียงของคุณ (.wav, .mp3, .flac, .m4a, .ogg)

In [ ]:
from google.colab import files
import IPython.display as ipd

uploaded = files.upload()

for filename in uploaded.keys():
    print("\n" + "=" * 55)
    print(f"FILE: {filename}")
    print("=" * 55)

    # เล่นเสียง
    ipd.display(ipd.Audio(filename))

    # วิเคราะห์
    prob, label = predict_cough(filename)
    if prob is not None:
        print(f"\n  TB Probability: {prob:.4f}")
        print(f"  RESULT: {label}")
        print()

        # แสดง Mel spectrogram
        import matplotlib.pyplot as plt
        y_seg = load_and_segment(filename)
        mel_spec, _ = make_mel_rgb(y_seg)

        plt.figure(figsize=(10, 4))
        plt.imshow(mel_spec, aspect='auto', origin='lower', cmap='magma')
        plt.colorbar(label='dB')
        plt.title(f"Mel Spectrogram — {filename}")
        plt.xlabel('Time')
        plt.ylabel('Mel Band')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  ERROR: {label}")

## 5. Batch Inference (หลายไฟล์)

รัน cell นี้ถ้าต้องการวิเคราะห์หลายไฟล์พร้อมกัน

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

results = []
print("\n" + "=" * 60)
print(f"{'FILE':<35} {'TB PROB':<10} {'RESULT'}")
print("=" * 60)

for filename in sorted(uploaded.keys()):
    prob, label = predict_cough(filename)
    if prob is not None:
        short = filename[:33] + ".." if len(filename) > 35 else filename
        tag = "TB ⚠️" if prob > 0.5 else "OK ✅"
        print(f"{short:<35} {prob:<10.4f} {tag}")
        results.append({"file": filename, "probability": prob, "is_tb": prob > 0.5})
    else:
        print(f"{filename:<35} {'ERROR':<10} {label}")

print("=" * 60)

if results:
    tb_count = sum(1 for r in results if r["is_tb"])
    avg_prob = np.mean([r["probability"] for r in results])
    print(f"\nสรุป: {tb_count}/{len(results)} ไฟล์  detected TB")
    print(f"      Average probability: {avg_prob:.4f}")

    # Export CSV
    import csv
    with open("cough_tb_results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file", "probability", "is_tb"])
        w.writeheader()
        w.writerows(results)
    print(f"\n✅ Export: cough_tb_results.csv")

---
## หมายสำคัญ

- Model นี้ train ด้วย **CODA TB DREAM Challenge** dataset
- Performance: Sensitivity ~85%, Specificity ~90% (จาก dev set)
- **เป็น screening tool → ไม่ใช่ diagnosis**
- ใช้ได้กับไฟล์ .wav, .mp3, .flac, .m4a, .ogg

### ขั้นตอนถัดไป
1. ทดสอบกับเสียงไอจริง
2. ถ้าได้ผลดี → export เป็น ONNX → ใส่ใน mobile app
3. ระหว่างนั้นก็ขอ Synapse access เพื่อเทรน model ของตัวเอง